## Data Engineering Module 4 - Medallion Architecture with Delta Lake

![med_new](./Assets/med_new.png)

### Loading CSV files into the Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["2019_edited.csv", "2020_edited.csv", "2021_edited.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading the CSV files into a Dataframe

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv')
display(df.limit(100))

### Defining the Schema for the Bronze Layer Dataframe

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

bronzeSchema = StructType([
    StructField("SalesOrderNumber", StringType()),
    StructField("SalesOrderLineNumber", IntegerType()),
    StructField("OrderDate", DateType()),
    StructField("CustomerName", StringType()),
    StructField("Email", StringType()),
    StructField("Item", StringType()),
    StructField("Quantity", IntegerType()),
    StructField("UnitPrice", FloatType()),
    StructField("Tax", FloatType())
])

### Creating the Bronze Layer Dataframe

In [0]:
bronze_df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/*_edited.csv', format='csv', schema=bronzeSchema)
display(bronze_df.limit(100))

### Creating the Bronze Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dp750_workspace.Bronze;

### Storing the Bronze Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
bronzeDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/bronze_sales_orders'
bronze_df.write.format('delta').mode('overwrite').save(bronzeDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
bronze_df.write.format('delta').saveAsTable('Bronze.bronze_sales_orders')

### Creating the Silver Layer by Cleaning and Transforming the Data

### Creating the Silver Layer Dataframe

In [0]:
silver_df = spark.read.format("delta").table("Bronze.bronze_sales_orders")
silver_df.show(10)

### Cleaning the Silver Layer Dataframe

In [0]:
from pyspark.sql.functions import col

silver_df = silver_df.dropDuplicates()
silver_df = silver_df.withColumn('Tax', col('UnitPrice') * 0.08)
silver_df = silver_df.withColumn('Tax', col('Tax').cast("float"))

### Creating the Silver Schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dp750_workspace.Silver;

### Storing the Silver Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
silverDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/silver_sales_orders'
silver_df.write.format('delta').mode('overwrite').save(silverDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
silver_df.write.format('delta').saveAsTable('Silver.silver_sales_orders')

### Creating the Gold Layer by Aggregating the Data

### Creating the Gold Layer Dataframe

In [0]:
gold_df = spark.read.format("delta").table("Silver.silver_sales_orders")
gold_df.show(10)

### Aggregating Yearly Sales Data

In [0]:
yearlySales = gold_df.select(year("OrderDate").alias("Year")).groupBy("Year").count().orderBy("Year")
display(yearlySales)

### Creating the Gold Schema in Unity Catalog

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS dp750_workspace.Gold;

### Storing the Gold Layer as a Delta Table

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Storing the Bronze Layer as a Delta Table in the Databricks File System (DBFS)
goldDeltaTablePath = f'/Volumes/{catalog_name}/default/spark_lab/delta/gold_sales_orders'
yearlySales.write.format('delta').mode('overwrite').save(goldDeltaTablePath)

# Storing the Bronze Layer as a Delta Table in the Data Catalog
yearlySales.write.format('delta').saveAsTable('Gold.gold_sales_orders')